# TOST drop-in (re-scoped): formal equivalence of the CIFAR-10 logistic vs Mahalanobis




In [ ]:
# ===================== [PREAMBLE] =====================
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import LedoitWolf

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
def make_pp(ds):
    mean=torch.tensor(CIFAR_MEAN).view(1,3,1,1).to(device); std=torch.tensor(CIFAR_STD).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    ck=(_find('resnet50_cifar10_finetuned.pt') or [None])[0]
    m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
    m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75): return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def median3(x):
    p=F.pad(x,(1,1,1,1),mode='reflect')
    return p.unfold(2,3,1).unfold(3,3,1).contiguous().view(*x.shape,9).median(-1).values
def feat_hfe(b): return ((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).detach().cpu().numpy()
def feat_gl(b,bb,pp,glsig):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); p1=F.softmax(bb(pp(gb(b,glsig))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def feat_predl1(b,bb,pp):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); sq=median3(jpeg_batch(b)).clamp(0,255); p2=F.softmax(bb(pp(sq)),1)
    return (p0-p2).abs().sum(1).cpu().numpy()

class Feats:
    def __init__(self, model):
        self.model=model; self.f={}
        self.h=[model.layer1.register_forward_hook(self._mk('l1')),
                model.layer2.register_forward_hook(self._mk('l2')),
                model.layer3.register_forward_hook(self._mk('l3')),
                model.layer4.register_forward_hook(self._mk('l4'))]
    def _mk(self,n):
        def hook(m,i,o): self.f[n]=o.mean((2,3)).detach()
        return hook
    def __call__(self,x):
        self.f={}
        with torch.no_grad(): self.model(x)
        return [self.f['l1'],self.f['l2'],self.f['l3'],self.f['l4']]
def extract_deep(imgs, fe, pp, bs=64):
    L=[[],[],[],[]]
    for i in range(0,len(imgs),bs):
        b=imgs[i:i+bs].to(device); fl=fe(pp(b))
        for k in range(4): L[k].append(fl[k].cpu().numpy())
    return [np.concatenate(x,0) for x in L]

def fit_maha_cc(Fl, labels):
    labels=np.asarray(labels); cls=np.unique(labels)
    assert len(cls)>1, 'calibration has <=1 class; predicted labels should give ~10 on CIFAR-10'
    Ws=[]; WMs=[]
    for f in Fl:
        mu=np.stack([f[labels==c].mean(0) for c in cls])
        cen=np.concatenate([f[labels==c]-mu[i] for i,c in enumerate(cls)],0)
        cov=LedoitWolf().fit(cen).covariance_; L=np.linalg.cholesky(np.linalg.inv(cov))
        Ws.append(L); WMs.append(mu.dot(L))
    return Ws, WMs, cls
def maha(Fl, Ws, WMs):
    s=0
    for f,L,WM in zip(Fl,Ws,WMs):
        w=f.dot(L); d2=(w**2).sum(1)[:,None]+(WM**2).sum(1)[None,:]-2.0*w.dot(WM.T); s=s+(-d2.min(1))
    return s
def find_mixed():
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: return p
    return None
def half(n, seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]
print('[PREAMBLE] ready')


In [ ]:
# ===================== build logistic & Mahalanobis test scores (CIFAR-10, unified 3-kind) =====================
glsig=0.5
bb=load_backbone('CIFAR-10'); pp=make_pp('CIFAR-10'); fe=Feats(bb)
mixed=pickle.load(open(find_mixed(),'rb'))
clean=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
adv  =[to224(im).cpu() for (im,lb,atk) in mixed if atk!='clean']
rng=np.random.RandomState(SEED); clean=[clean[i] for i in rng.permutation(len(clean))[:500]]
Xc=torch.cat(clean,0); Xa=torch.cat(adv,0)
ci,ti=half(len(clean)); Xcal=Xc[ci]; Xte=Xc[ti]
# predicted labels for class-conditional Mahalanobis calibration
ycal=[]
for i in range(0,len(Xcal),64):
    b=Xcal[i:i+64].to(device)
    with torch.no_grad(): ycal.append(bb(pp(b)).argmax(1).cpu().numpy())
ycal=np.concatenate(ycal)
# unified 3-kind hard negatives on the clean test half
noise=(Xte+torch.randn_like(Xte)*8.0).clamp(0,255); jp=jpeg_batch(Xte.to(device)).cpu()
bl=gb(Xte.to(device),1.0).clamp(0,255).cpu(); Xhn=torch.cat([noise,jp,bl],0)
ai_c,ai_t=half(len(adv),seed=1); hi_c,hi_t=half(Xhn.shape[0],seed=2)

# ---- z-features and standardization (clean calibration) ----
def scal(X):
    H=[];G=[];P=[]
    for i in range(0,len(X),64):
        b=X[i:i+64].to(device); H.append(feat_hfe(b)); G.append(feat_gl(b,bb,pp,glsig)); P.append(feat_predl1(b,bb,pp))
    return np.concatenate(H),np.concatenate(G),np.concatenate(P)
Hc,Gc,Pc=scal(Xcal); muH,sdH=Hc.mean(),Hc.std()+1e-8; muG,sdG=Gc.mean(),Gc.std()+1e-8; muP,sdP=Pc.mean(),Pc.std()+1e-8
def Zof(X):
    H,G,P=scal(X); return np.stack([(H-muH)/sdH,(G-muG)/sdG,(P-muP)/sdP],1)
Zc=Zof(Xcal); Zte=Zof(Xte); Za=Zof(Xa); Zh=Zof(Xhn)

# ---- supervised logistic, trained leakage-safe on the TRAIN split only ----
Xtr=np.concatenate([Zc,Zh[hi_c],Za[ai_c]],0); ytr=np.r_[np.zeros(len(Zc)+len(hi_c)),np.ones(len(ai_c))]
lr=LogisticRegression(max_iter=2000,class_weight='balanced').fit(Xtr,ytr)
prob=lambda Zx: lr.predict_proba(Zx)[:,1]
log_neg=np.concatenate([prob(Zte),prob(Zh[hi_t])]); log_pos=prob(Za[ai_t])

# ---- class-conditional Mahalanobis on the same test set ----
Ws,WMs,cls=fit_maha_cc(extract_deep(Xcal,fe,pp), ycal); print('Maha calib classes:', len(cls))
Fte=extract_deep(Xte,fe,pp); Fhn=extract_deep(Xhn,fe,pp); Fa=extract_deep(Xa,fe,pp)
mah_neg=np.concatenate([-maha(Fte,Ws,WMs), -maha([f[hi_t] for f in Fhn],Ws,WMs)]); mah_pos=-maha([f[ai_t] for f in Fa],Ws,WMs)

def auroc(neg,pos): return roc_auc_score(np.r_[np.zeros(len(neg)),np.ones(len(pos))], np.r_[neg,pos])
# median (context only)
muS=np.median(np.stack([(Hc-muH)/sdH,(Gc-muG)/sdG,(Pc-muP)/sdP],1),1).mean()
med=lambda Z: np.abs(np.median(Z,1)-muS)
med_neg=np.concatenate([med(Zte),med(Zh[hi_t])]); med_pos=med(Za[ai_t])
print(f'logistic AUROC={auroc(log_neg,log_pos):.4f}  Mahalanobis AUROC={auroc(mah_neg,mah_pos):.4f}  '
      f'(median AUROC={auroc(med_neg,med_pos):.4f}, context only)')
print(f'observed diff (logistic - Maha) = {auroc(log_neg,log_pos)-auroc(mah_neg,mah_pos):+.4f}')
print('logistic weights [z_HF,z_GL,z_PL]:', np.round(lr.coef_[0],3).tolist())


In [ ]:
# ===================== paired bootstrap + TOST (logistic vs Mahalanobis) =====================
B=5000; MARGINS=[0.02,0.03,0.05]
assert len(log_neg)==len(mah_neg) and len(log_pos)==len(mah_pos)
nN=len(log_neg); nP=len(log_pos); y=np.r_[np.zeros(nN),np.ones(nP)]
obs=auroc(log_neg,log_pos)-auroc(mah_neg,mah_pos)
rng=np.random.RandomState(SEED); D=np.empty(B)
for b in range(B):
    ni=rng.randint(0,nN,nN); pi=rng.randint(0,nP,nP)
    lb=roc_auc_score(y,np.r_[log_neg[ni],log_pos[pi]]); hb=roc_auc_score(y,np.r_[mah_neg[ni],mah_pos[pi]])
    D[b]=lb-hb
p2_5,p5,p95,p97_5=np.percentile(D,[2.5,5,95,97.5])
dstar_equiv=float(max(abs(p5),abs(p95))); dstar_noninf=float(max(0.0,-p5))
per_margin={f'{d}':{'equivalence':bool(p5>-d and p95<d),'non_inferiority':bool(p5>-d)} for d in MARGINS}

print('=== TOST / non-inferiority (CIFAR-10, logistic - Mahalanobis) ===')
print(f'observed diff {obs:+.4f} | 90% CI [{p5:+.4f}, {p95:+.4f}] | 95% CI [{p2_5:+.4f}, {p97_5:+.4f}]')
print(f'smallest supportable margins: equivalence delta* = {dstar_equiv:.4f} | non-inferiority delta* = {dstar_noninf:.4f}')
for d in MARGINS:
    m=per_margin[f'{d}']
    print(f'  margin {d:.2f}: equivalence={"HOLDS" if m["equivalence"] else "FAILS"} | non-inferiority={"HOLDS" if m["non_inferiority"] else "FAILS"}')

OUT='./tost_results'; os.makedirs(OUT,exist_ok=True)
json.dump({'cifar10_logistic_vs_maha':{
    'auroc_logistic':round(float(auroc(log_neg,log_pos)),4),'auroc_maha':round(float(auroc(mah_neg,mah_pos)),4),
    'auroc_median_context':round(float(auroc(med_neg,med_pos)),4),
    'observed_diff':round(float(obs),4),'ci90':[round(float(p5),4),round(float(p95),4)],
    'ci95':[round(float(p2_5),4),round(float(p97_5),4)],
    'dstar_equivalence':round(dstar_equiv,4),'dstar_noninferiority':round(dstar_noninf,4),
    'margins':per_margin,'B':B,'n_neg':int(nN),'n_pos':int(nP),'maha_classes':int(len(cls)),
    'hard_negatives':'noise+jpeg+blur'}},
    open(os.path.join(OUT,'tost_logistic_cifar10.json'),'w'), indent=2)
print('\nsaved', os.path.join(OUT,'tost_logistic_cifar10.json'))
